# Notebook 03 — SVD Matrix Factorization Baseline

SVD collaborative filtering baseline. Builds a sparse mean-centered user-item matrix from training reviews, applies truncated SVD, and evaluates ranking quality on the validation and test sets.

**Metrics:** Hit@K, NDCG@K, Precision@K, Recall@K for K = 5, 10, 20

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import collections
import os

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import svds

## Load Data

In [ ]:
DATA_DIR = "/content/drive/MyDrive/rec_system"

train = pd.read_parquet(f"{DATA_DIR}/train_reviews.parquet", columns=["user_id", "business_id", "stars_normalized"])
val   = pd.read_parquet(f"{DATA_DIR}/val_reviews.parquet",   columns=["user_id", "business_id", "label"])
test  = pd.read_parquet(f"{DATA_DIR}/test_reviews.parquet",  columns=["user_id", "business_id", "label"])

print(f"Train: {len(train):,} rows | {train['user_id'].nunique():,} users | {train['business_id'].nunique():,} businesses")
print(f"Val:   {len(val):,} rows")
print(f"Test:  {len(test):,} rows")

## Build User-Item Matrix

In [ ]:
# Encode raw string IDs to contiguous integer indices (train vocab only)
user2idx = {u: i for i, u in enumerate(train["user_id"].unique())}
item2idx = {b: i for i, b in enumerate(train["business_id"].unique())}

n_users = len(user2idx)
n_items = len(item2idx)
print(f"Matrix shape: {n_users:,} users x {n_items:,} items")

In [ ]:
# Build CSR sparse matrix: rows=users, cols=items, values=stars_normalized
# Mean-centered so SVD captures relative preference, not per-user rating bias
user_idx = train["user_id"].map(user2idx).values
item_idx = train["business_id"].map(item2idx).values
ratings  = train["stars_normalized"].astype(np.float32).values

R = sp.csr_matrix((ratings, (user_idx, item_idx)), shape=(n_users, n_items))
print(f"R: {R.shape}, nnz={R.nnz:,}, density={R.nnz / (R.shape[0] * R.shape[1]):.5%}")

## SVD Factorization

In [ ]:
N_FACTORS = 100  # default; see factor sweep section below

# svds returns singular values in ascending order — reverse to descending
U, sigma, Vt = svds(R.astype(np.float64), k=N_FACTORS)
U     = U[:, ::-1]
sigma = sigma[::-1]
Vt    = Vt[::-1, :]

print(f"U: {U.shape}, sigma: {sigma.shape}, Vt: {Vt.shape}")
print(f"Top-5 singular values: {sigma[:5].round(1)}")

In [ ]:
# Precompute item-side factors: shape (n_factors, n_items)
# Per-user scores = U[uid] @ item_factors  — never materialise the full dense matrix
item_factors = np.diag(sigma) @ Vt

## Evaluation

In [ ]:
def hit_at_k(rank: int, k: int) -> float:
    return 1.0 if rank <= k else 0.0

def ndcg_at_k(rank: int, k: int) -> float:
    return 1.0 / np.log2(rank + 1) if rank <= k else 0.0

def precision_at_k(rank: int, k: int) -> float:
    return (1.0 / k) if rank <= k else 0.0

def recall_at_k(rank: int, k: int) -> float:
    return 1.0 if rank <= k else 0.0

In [ ]:
# Build per-user set of training item indices (to exclude from candidate ranking)
train_items_per_user = collections.defaultdict(set)
for uid, bid in zip(train["user_id"].values, train["business_id"].values):
    train_items_per_user[uid].add(item2idx[bid])

In [ ]:
def evaluate(split_df, split_name, U_mat=None, if_mat=None, n_factors=None, Ks=(5, 10, 20)):
    """Leave-one-out ranking evaluation. Only evaluates users with label==1.

    Batched for performance: scores all items per user in groups of BATCH rows.
    Target item is excluded from the training mask to handle duplicate reviews.
    """
    _U  = U_mat     if U_mat     is not None else U
    _if = if_mat    if if_mat    is not None else item_factors
    _k  = n_factors if n_factors is not None else N_FACTORS

    # Only rank positive held-out items; skip label=0 (no ground-truth positive)
    eval_df = split_df[
        (split_df["label"] == 1)
        & split_df["user_id"].isin(user2idx)
        & split_df["business_id"].isin(item2idx)
    ].reset_index(drop=True)

    n_skipped = len(split_df) - len(eval_df)
    print(f"{split_name}: {len(eval_df):,} evaluable users ({n_skipped:,} skipped — label=0 or OOV)")

    user_indices = eval_df["user_id"].map(user2idx).values
    item_indices = eval_df["business_id"].map(item2idx).values
    uid_strings  = eval_df["user_id"].values

    all_ranks = np.empty(len(eval_df), dtype=np.int32)
    BATCH = 1000

    for start in range(0, len(eval_df), BATCH):
        end   = min(start + BATCH, len(eval_df))
        bu    = user_indices[start:end]   # batch user indices
        bi    = item_indices[start:end]   # batch target item indices
        bs    = uid_strings[start:end]    # batch user id strings

        scores = _U[bu] @ _if             # (batch, n_items)

        for i, uid_str in enumerate(bs):
            # Mask train items but preserve the target item.
            # Preserving prevents a duplicate-review edge case where a user
            # reviewed the same business in both train and val/test, which
            # would otherwise mask the target and assign worst-possible rank.
            seen = list(train_items_per_user[uid_str] - {bi[i]})
            if seen:
                scores[i, seen] = -np.inf

        target_scores = scores[np.arange(end - start), bi]   # (batch,)
        all_ranks[start:end] = (scores > target_scores[:, np.newaxis]).sum(axis=1) + 1

    results = {}
    for k in Ks:
        hits = (all_ranks <= k).astype(float)
        results[k] = {
            "Hit@K":       hits.mean(),
            "NDCG@K":      np.where(all_ranks <= k, 1.0 / np.log2(all_ranks + 1), 0.0).mean(),
            "Precision@K": np.where(all_ranks <= k, 1.0 / k, 0.0).mean(),
            "Recall@K":    hits.mean(),   # with 1 positive, recall == hit
        }

    rows = [{"K": k, **results[k]} for k in Ks]
    metrics_df = pd.DataFrame(rows).set_index("K")
    print(f"\n{split_name} Results (n_factors={_k}):")
    print(metrics_df.round(4).to_string())
    return metrics_df

In [ ]:
val_metrics  = evaluate(val,  "Validation")
test_metrics = evaluate(test, "Test")

## Results

In [ ]:
import matplotlib.pyplot as plt

Ks = [5, 10, 20]
metric_cols = ["Hit@K", "NDCG@K", "Precision@K", "Recall@K"]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col in zip(axes, metric_cols):
    ax.plot(val_metrics.index,  val_metrics[col],  marker="o", label="Val")
    ax.plot(test_metrics.index, test_metrics[col], marker="s", label="Test")
    ax.set_title(col)
    ax.set_xlabel("K")
    ax.set_xticks(Ks)
    ax.legend()

plt.suptitle(f"SVD Baseline — n_factors={N_FACTORS}", fontsize=13)
plt.tight_layout()
plt.show()

## Save Results

In [ ]:
results_dir = f"{DATA_DIR}/results"
os.makedirs(results_dir, exist_ok=True)

val_metrics.to_csv(f"{results_dir}/svd_val_metrics_k{N_FACTORS}.csv")
test_metrics.to_csv(f"{results_dir}/svd_test_metrics_k{N_FACTORS}.csv")

print(f"Saved val metrics  -> {results_dir}/svd_val_metrics_k{N_FACTORS}.csv")
print(f"Saved test metrics -> {results_dir}/svd_test_metrics_k{N_FACTORS}.csv")

## Factor Sweep (optional — run to select n_factors)

Sweeps `n_factors` over [50, 100, 200] and reports NDCG@K on the validation set. Use to pick the best `N_FACTORS` before recording final test metrics.

In [ ]:
sweep_results = {}

for k in [50, 100, 200]:
    U_k, sigma_k, Vt_k = svds(R.astype(np.float64), k=k)
    U_k     = U_k[:, ::-1]
    sigma_k = sigma_k[::-1]
    Vt_k    = Vt_k[::-1, :]
    if_k    = np.diag(sigma_k) @ Vt_k

    m = evaluate(val, f"Val k={k}", U_mat=U_k, if_mat=if_k, n_factors=k)
    sweep_results[k] = m

sweep_df = pd.concat(
    {f"k={k}": sweep_results[k]["NDCG@K"] for k in sweep_results},
    axis=1
)
print("\nNDCG@K across factor counts:")
print(sweep_df.round(4))